In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a weight-only INT4 quantized matrix multiplication (W4A16), a core kernel used in
  modern LLM inference. Given a float16 activation matrix <code>x</code> of shape
  <code>M &times; K</code> and a weight matrix stored in packed INT4 format, compute the output
  matrix <code>y = x &times; W<sup>T</sup></code> of shape <code>M &times; N</code>, where
  <code>W</code> is the dequantized float16 weight matrix of shape <code>N &times; K</code>.
</p>

<p>
  <strong>Packing format:</strong> Each byte of <code>w_q</code> stores two INT4 weights. The
  high nibble (bits 7&ndash;4) holds weight <code>w[n, 2i]</code> and the low nibble (bits
  3&ndash;0) holds <code>w[n, 2i+1]</code>. INT4 values are stored unsigned in the range
  [0,&nbsp;15] with an offset of 8, so the signed weight is <code>nibble&nbsp;&minus;&nbsp;8</code>,
  giving values in [&minus;8,&nbsp;7].
</p>

<p>
  <strong>Dequantization:</strong> Weights are dequantized group-wise. Each contiguous block of
  <code>group_size</code> weights along the <code>K</code> dimension shares one float16 scale:
</p>
<pre>
W[n, k] = (w_q_nibble[n, k] - 8) * scales[n, k // group_size]
</pre>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in <code>y</code></li>
</ul>

<h2>Example</h2>
<p>
  Input (<code>M</code> = 2, <code>N</code> = 4, <code>K</code> = 4, <code>group_size</code> = 2):
</p>
<p>
  Activations $x$ (float16, $2 \times 4$):
  $$
  \begin{bmatrix}
  1.0 & 0.0 & 1.0 & 0.0 \\
  0.0 & 1.0 & 0.0 & 1.0
  \end{bmatrix}
  $$
  Packed weights $w\_q$ (uint8, $4 \times 2$) with signed INT4 values in brackets:
  $$
  \begin{bmatrix}
  \texttt{0x99} & \texttt{0x99} \\
  \texttt{0xAA} & \texttt{0xAA} \\
  \texttt{0x77} & \texttt{0x77} \\
  \texttt{0x88} & \texttt{0x88}
  \end{bmatrix}
  \;\Rightarrow\;
  W_{\text{int4}} =
  \begin{bmatrix}
  1 & 1 & 1 & 1 \\
  2 & 2 & 2 & 2 \\
  -1 & -1 & -1 & -1 \\
  0 & 0 & 0 & 0
  \end{bmatrix}
  $$
  Scales (float16, $4 \times 2$, all entries 0.5):
  $$
  \begin{bmatrix}
  0.5 & 0.5 \\
  0.5 & 0.5 \\
  0.5 & 0.5 \\
  0.5 & 0.5
  \end{bmatrix}
  \;\Rightarrow\;
  W_{\text{dequant}} =
  \begin{bmatrix}
  0.5 & 0.5 & 0.5 & 0.5 \\
  1.0 & 1.0 & 1.0 & 1.0 \\
  -0.5 & -0.5 & -0.5 & -0.5 \\
  0.0 & 0.0 & 0.0 & 0.0
  \end{bmatrix}
  $$
  Output $y = x \times W^T$ (float16, $2 \times 4$):
  $$
  \begin{bmatrix}
  1.0 & 2.0 & -1.0 & 0.0 \\
  1.0 & 2.0 & -1.0 & 0.0
  \end{bmatrix}
  $$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>M</code>, <code>N</code> &le; 8,192</li>
  <li>1 &le; <code>K</code> &le; 8,192</li>
  <li><code>K</code> is divisible by <code>2</code> and by <code>group_size</code></li>
  <li><code>group_size</code> &isin; {2, 4, 8, 16, 32, 64, 128}</li>
  <li>All tensors are stored in row-major order</li>
  <li>Input dtype: <code>x</code> and <code>scales</code> are float16; <code>w_q</code> is uint8</li>
  <li>Output dtype: <code>y</code> is float16</li>
  <li>Performance is measured with <code>M</code> = 4,096, <code>N</code> = 4,096, <code>K</code> = 4,096, <code>group_size</code> = 128</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_fp16.h>
#include <cuda_runtime.h>
#include <stdint.h>

// x, w_q, scales, y are device pointers
extern "C" void solve(const __half* x, const uint8_t* w_q, const __half* scales, __half* y, int M,
                      int N, int K, int group_size) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# x, w_q, scales, y are tensors on the GPU
@cute.jit
def solve(
    x: cute.Tensor,
    w_q: cute.Tensor,
    scales: cute.Tensor,
    y: cute.Tensor,
    M: cute.Int32,
    N: cute.Int32,
    K: cute.Int32,
    group_size: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# x, w_q, scales are tensors on GPU
@jax.jit
def solve(
    x: jax.Array,
    w_q: jax.Array,
    scales: jax.Array,
    M: int,
    N: int,
    K: int,
    group_size: int,
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.memory import UnsafePointer


# x, w_q, scales, y are device pointers
@export
def solve(
    x: UnsafePointer[Float16, MutExternalOrigin],
    w_q: UnsafePointer[UInt8, MutExternalOrigin],
    scales: UnsafePointer[Float16, MutExternalOrigin],
    y: UnsafePointer[Float16, MutExternalOrigin],
    M: Int32,
    N: Int32,
    K: Int32,
    group_size: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# x, w_q, scales, y are tensors on the GPU
def solve(
    x: torch.Tensor,
    w_q: torch.Tensor,
    scales: torch.Tensor,
    y: torch.Tensor,
    M: int,
    N: int,
    K: int,
    group_size: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# x, w_q, scales, y are tensors on the GPU
def solve(
    x: torch.Tensor,
    w_q: torch.Tensor,
    scales: torch.Tensor,
    y: torch.Tensor,
    M: int,
    N: int,
    K: int,
    group_size: int,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="INT4 Weight-Only Quantized MatMul",
            atol=1e-02,
            rtol=1e-02,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        x: torch.Tensor,
        w_q: torch.Tensor,
        scales: torch.Tensor,
        y: torch.Tensor,
        M: int,
        N: int,
        K: int,
        group_size: int,
    ):
        assert x.shape == (M, K)
        assert w_q.shape == (N, K // 2)
        assert scales.shape == (N, K // group_size)
        assert y.shape == (M, N)
        assert x.dtype == torch.float16
        assert w_q.dtype == torch.uint8
        assert scales.dtype == torch.float16
        assert y.dtype == torch.float16
        assert x.device.type == "cuda"
        assert w_q.device.type == "cuda"
        assert scales.device.type == "cuda"
        assert y.device.type == "cuda"

        # Unpack INT4 weights from packed uint8 bytes.
        # w_q[n, i] stores two weights: w[n, 2*i] in the high nibble (bits 7:4)
        # and w[n, 2*i+1] in the low nibble (bits 3:0).
        # INT4 values are stored unsigned (0–15) with an offset of 8,
        # so the signed value is nibble - 8, giving range [-8, 7].
        w_high = ((w_q >> 4) & 0xF).to(torch.int32) - 8  # [N, K//2]
        w_low = (w_q & 0xF).to(torch.int32) - 8  # [N, K//2]

        # Interleave high and low nibbles to reconstruct [N, K]
        w_int = torch.stack([w_high, w_low], dim=-1).reshape(N, K)  # [N, K]

        # Apply group-wise scales: dequantize each group
        n_groups = K // group_size
        w_groups = w_int.reshape(N, n_groups, group_size).float()  # [N, n_groups, group_size]
        scales_f = scales.float().unsqueeze(-1)  # [N, n_groups, 1]
        w_dequant = (w_groups * scales_f).reshape(N, K)  # [N, K]

        # MatMul: x [M, K] @ w_dequant.T [K, N] = y [M, N]
        y.copy_((x.float() @ w_dequant.T).half())

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "x": (ctypes.POINTER(ctypes.c_uint16), "in"),
            "w_q": (ctypes.POINTER(ctypes.c_uint8), "in"),
            "scales": (ctypes.POINTER(ctypes.c_uint16), "in"),
            "y": (ctypes.POINTER(ctypes.c_uint16), "out"),
            "M": (ctypes.c_int, "in"),
            "N": (ctypes.c_int, "in"),
            "K": (ctypes.c_int, "in"),
            "group_size": (ctypes.c_int, "in"),
        }

    def _make_test_case(self, M: int, N: int, K: int, group_size: int, zero_x: bool = False):
        device = "cuda"
        if zero_x:
            x = torch.zeros(M, K, device=device, dtype=torch.float16)
        else:
            x = torch.randn(M, K, device=device, dtype=torch.float16)
        # Random packed INT4 weights: each byte holds two nibbles in [0,15]
        w_q = torch.randint(0, 256, (N, K // 2), dtype=torch.uint8, device=device)
        # Small positive scales to keep magnitudes reasonable
        scales = torch.rand(N, K // group_size, device=device, dtype=torch.float16) * 0.1 + 0.01
        y = torch.empty(M, N, device=device, dtype=torch.float16)
        return {
            "x": x,
            "w_q": w_q,
            "scales": scales,
            "y": y,
            "M": M,
            "N": N,
            "K": K,
            "group_size": group_size,
        }

    def generate_example_test(self) -> Dict[str, Any]:
        device = "cuda"
        M, N, K, group_size = 2, 4, 4, 2

        x = torch.tensor(
            [[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 1.0]],
            device=device,
            dtype=torch.float16,
        )
        # Packed INT4 weights (high nibble first).
        # Row 0: weights [1,1,1,1]  → nibbles stored as [9,9,9,9] → bytes [0x99, 0x99] = [153, 153]
        # Row 1: weights [2,2,2,2]  → nibbles [10,10,10,10]      → bytes [0xAA, 0xAA] = [170, 170]
        # Row 2: weights [-1,-1,-1,-1] → nibbles [7,7,7,7]       → bytes [0x77, 0x77] = [119, 119]
        # Row 3: weights [0,0,0,0]  → nibbles [8,8,8,8]          → bytes [0x88, 0x88] = [136, 136]
        w_q = torch.tensor(
            [[153, 153], [170, 170], [119, 119], [136, 136]],
            dtype=torch.uint8,
            device=device,
        )
        # One scale per group (group_size=2 → 2 groups per row), all 0.5
        scales = torch.full((N, K // group_size), 0.5, device=device, dtype=torch.float16)
        y = torch.empty(M, N, device=device, dtype=torch.float16)

        return {
            "x": x,
            "w_q": w_q,
            "scales": scales,
            "y": y,
            "M": M,
            "N": N,
            "K": K,
            "group_size": group_size,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        torch.manual_seed(42)
        tests = []

        # Edge cases — tiny K, small group_size
        tests.append(self._make_test_case(1, 2, 4, 2, zero_x=True))
        tests.append(self._make_test_case(2, 4, 4, 2))
        tests.append(self._make_test_case(3, 5, 8, 4))

        # Power-of-2 sizes
        tests.append(self._make_test_case(16, 16, 32, 16))
        tests.append(self._make_test_case(32, 64, 64, 32))
        tests.append(self._make_test_case(64, 128, 128, 64))

        # Non-power-of-2 sizes
        tests.append(self._make_test_case(30, 50, 64, 32))
        tests.append(self._make_test_case(100, 200, 128, 64))
        tests.append(self._make_test_case(255, 100, 128, 64))

        # Realistic LLM inference sizes
        tests.append(self._make_test_case(128, 256, 512, 128))

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        torch.manual_seed(0)
        # Typical LLM weight matrix: 4096×4096 with group_size=128
        return self._make_test_case(4096, 4096, 4096, 128)


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
